In [3]:
import os
import numpy as np

midi_file_paths = []
for root, dirs, files in os.walk("/Users/christofferkassoe/Documents/KogDat/EM3/Behavioral/POP909-Dataset/POP909"):
    for file in files:
        if file.endswith(".mid") and not root.endswith("versions"):
            midi_file_paths.append(os.path.join(root, file))

print(f"\nTotal MIDI files found: {len(midi_file_paths)}")


Total MIDI files found: 909


In [ ]:
# Midi extraction and conversion functions
import mido

def ExtractMelodyTones(file_path):

    abs_path = file_path

    midi_mel = mido.MidiFile(abs_path)
    seq = []

    for msg in midi_mel.tracks[1]: #Vi vil kun have melodistemen
        if msg.type == 'note_on' and msg.velocity > 0:  # Kun note_on with velocity > 0
            if seq == []:
                seq.append(msg.note)
            elif msg.note != seq[-1]:
                seq.append(msg.note)

    return seq

def ConvertToFreqs(seq):
    return [round(440 * (2**((tone - 69)/12)), 3) for tone in seq]

melodies_midi = [ExtractMelodyTones(path) for path in midi_file_paths]
melodies_freqs = [ConvertToFreqs(seq) for seq in melodies_midi]

In [ ]:
# Print some statistics about the melodies
print("There is " + str(len(melodies_freqs)) + " sequences")
print("with an average length of " + str(sum([len(seq) for seq in melodies_freqs]) / len(melodies_freqs)) + " tones")

individual_tones = set()
for seq in melodies_midi:
    for tone in seq:
        individual_tones.add(tone)

print("Consisting of " + str(len(individual_tones)) + " unique tones")

There is 909 sequences
with an average length of 260.1771177117712 tones
Consisting of 55 unique tones


To test out different models, we try out both the:

### Variable Order Markov Model


First we need the n-grams the maximum order of seven (septogram)

In [6]:
n_grams = []
max_order = 7

for i in range(max_order+1):
    i_gram = []
    if i == 0:
        for seq in melodies_midi:
            for tone in seq:
                i_gram.append(tuple([tone]))
    else:
        for seq in melodies_midi:
            for j, tone in enumerate(seq[:-i]):
                i_gram.append(tuple(seq[j:j+i+1]))

    n_grams.append(i_gram)



In [7]:
for i, n_gram in enumerate(n_grams):
    print("There is " + str(len(n_gram)) + " " + str(i) + "-grams in the corpus")

There is 236501 0-grams in the corpus
There is 235592 1-grams in the corpus
There is 234683 2-grams in the corpus
There is 233774 3-grams in the corpus
There is 232865 4-grams in the corpus
There is 231956 5-grams in the corpus
There is 231047 6-grams in the corpus
There is 230138 7-grams in the corpus


In [8]:
from collections import defaultdict, Counter

n_gram_counts = [defaultdict(int) for _ in range(8)]

for order, i_grams in enumerate(n_grams):
    for gram in i_grams:
        n_gram_counts[order][gram] += 1 

Calculates the probabilities

$$
p_{esc}=number of distinct next symbols/total count for this context​
$$

In [9]:
def PredictNext(seq):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = len(seq)
    num_tones = 5

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob
    
    return dict(sorted(probs.items(), key=lambda x: -x[1])[:num_tones])


Generating a simple melodi with pure probability

In [10]:
def ToneRandChoice(probs):
    total = sum(probs.values())
    choice = np.random.random() * total
    for key, value in probs.items():
        if choice < value:
            return key, value
        choice -= value


def GenerateMelody(probe, length):
    context = (probe,)
    for i in range(length):
        pred = PredictNext(context)
        new_tone, _ = ToneRandChoice(pred)
        context += (new_tone,)
    return list(context)

print(ConvertToFreqs(GenerateMelody(77,7)))

[698.456, 622.254, 466.164, 554.365, 622.254, 698.456, 466.164, 415.305]


Using this to create binary trees for sequential melody production
$$
N=2^0+2^1+...+2^{l-1}=2^l-1
$$

In [13]:
import math

def GenerateBinaryTree(probe, length):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    tree[0] = probe
    prob_tree[0] = 1
    for layer in range(1,length,1):
        fst_pos = 2**layer - 1
        for node in range(0,2**layer, 2):
            parents = [math.ceil((node+fst_pos)/2)-1]
            for i in range(layer-2):
                parents.append(math.ceil(parents[-1]/2) - 1)
            context = tuple([tree[i] for i in parents])
            pred = PredictNext(context)
            tone1, prob1 = ToneRandChoice(pred)
            pred.pop(tone1)
            tone2, prob2 = ToneRandChoice(pred)
            choice = np.random.random()
            if choice > 0.5:
                tone1, tone2 = tone2, tone1
                prob1, prob2 = prob2, prob1
            tree[fst_pos + node] = tone1
            tree[fst_pos + node + 1] = tone2

            prob_tree[fst_pos + node] = prob1
            prob_tree[fst_pos + node + 1] = prob2
    return tree, prob_tree

tones, probs = GenerateBinaryTree(67,5)     
print(tones)
print(probs)

[67, 72, 65, 70, 74, 62, 67, 75, 73, 70, 74, 62, 67, 63, 62, 68, 75, 73, 70, 74, 65, 74, 67, 62, 63, 67, 62, 62, 70, 62, 67]
[1, 0.05130196801521564, 0.21524032144490504, 0.21277668743608796, 0.23520385155262538, 0.08413112208109158, 0.30048255588218986, 0.16049731156101876, 0.09122784738628878, 0.20162854029987937, 0.34578811226817674, 0.2045256743834182, 0.4475275047677989, 0.23690632385841454, 0.14279236170810017, 0.13459271653791352, 0.2726610250266393, 0.8620300632528368, 0.05170346490979473, 0.2514960949334012, 0.045889254117840984, 0.34578811226817674, 0.054502336414889996, 0.2045256743834182, 0.06546673967630262, 0.3833249017549288, 0.3741010489563675, 0.10475806587899852, 0.03138881737335445, 0.2701228030954291, 0.36772665118921416]


Just for testing the tree, i will make a function to choose a random sequence

In [205]:
def TreeRandSeq(tree, length):
    seq = [tree[0]]
    parent = 0
    for i in range(length-1):
        choice = np.random.random()
        if choice > 0.5:
            seq.append(tree[2*(parent+1)-1])
            parent = 2*(parent+1)-1
        else:
            seq.append(tree[2*(parent+1)])
            parent = 2*(parent+1)
    return seq

SomeTree = ConvertToFreqs(GenerateBinaryTree(77,3))
print(SomeTree)
print(TreeRandSeq(SomeTree, 3))

[698.456, 830.609, 587.33, 739.989, 698.456, 698.456, 493.883]
[698.456, 830.609, 739.989]
